In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

pd.options.display.float_format = '{:.2f}'.format

HS_data = pd.read_csv('AICPA_regressionAnalysisData (1).csv')
HS_data['date'] = pd.to_datetime(HS_data['date'])

display(HS_data)


In [ ]:
HS_data['Spring_DV'] = np.where(HS_data['date'].dt.month.isin([3,4,5]),1,0)
HS_data['Summer_DV'] = np.where(HS_data['date'].dt.month.isin([6,7,8]),1,0)

display(HS_data)


In [ ]:
# interaction terms
HS_data['summer_interaction'] = HS_data['production'] * HS_data['Summer_DV']
HS_data['spring_interaction'] = HS_data['coolDD'] * HS_data['Spring_DV']

display(HS_data)


In [ ]:
dt4training = HS_data.loc[HS_data['type']=='dt4training']
dt4testing = HS_data.loc[HS_data['type']=='dt4testing']

display(dt4training)
display(dt4testing)


In [ ]:
# train model 1
y = dt4training.loc[:,'revenue']
x = dt4training.loc[:,['production','Summer_DV','summer_interaction']]
x = sm.add_constant(x)

model1 = sm.OLS(y,x).fit()

model1.params


In [ ]:
x_test = dt4testing.loc[:,['production','Summer_DV','summer_interaction']]
x_test = sm.add_constant(x_test, has_constant='add')

model1_assessment = dt4testing.copy()
model1_assessment['model1_predicted_revenue'] = model1.predict(x_test)
model1_assessment['model1_abs_pct_error'] = abs(
    (model1_assessment['revenue'] - model1_assessment['model1_predicted_revenue'])
    / model1_assessment['revenue']
)

display(model1_assessment)


In [ ]:
model1_mape = model1_assessment['model1_abs_pct_error'].mean()

model1_mape


In [ ]:
plt.figure(figsize=(10,6))

plt.title('model 1 revenue forecast')
plt.xlabel('date')
plt.ylabel('revenue')

plt.plot(model1_assessment['date'],model1_assessment['revenue'],marker='o',label='actual')
plt.plot(model1_assessment['date'],model1_assessment['model1_predicted_revenue'],marker='o',label='predicted')

plt.legend()
plt.xticks(rotation=45)
plt.show()


In [ ]:
# train model 2
y = dt4training.loc[:,'revenue']
x = dt4training.loc[:,['coolDD','Spring_DV','spring_interaction']]
x = sm.add_constant(x)

model2 = sm.OLS(y,x).fit()

model2.params


In [ ]:
x_test = dt4testing.loc[:,['coolDD','Spring_DV','spring_interaction']]
x_test = sm.add_constant(x_test, has_constant='add')

model2_assessment = dt4testing.copy()
model2_assessment['model2_predicted_revenue'] = model2.predict(x_test)
model2_assessment['model2_abs_pct_error'] = abs(
    (model2_assessment['revenue'] - model2_assessment['model2_predicted_revenue'])
    / model2_assessment['revenue']
)

display(model2_assessment)


In [ ]:
model2_mape = model2_assessment['model2_abs_pct_error'].mean()

model2_mape


In [ ]:
plt.figure(figsize=(10,6))

plt.title('model 2 revenue forecast')
plt.xlabel('date')
plt.ylabel('revenue')

plt.plot(model2_assessment['date'],model2_assessment['revenue'],marker='o',label='actual')
plt.plot(model2_assessment['date'],model2_assessment['model2_predicted_revenue'],marker='o',label='predicted')

plt.legend()
plt.xticks(rotation=45)
plt.show()


In [ ]:
model_comparison = pd.DataFrame({
    'model':['model 1','model 2'],
    'MAPE':[model1_mape,model2_mape]
})

display(model_comparison)


In [ ]:
plt.figure(figsize=(10,6))

plt.title('revenue forecasts')
plt.xlabel('date')
plt.ylabel('revenue')

plt.plot(dt4testing['date'],dt4testing['revenue'],marker='o',label='actual')
plt.plot(dt4testing['date'],model1_assessment['model1_predicted_revenue'],marker='o',label='model 1')
plt.plot(dt4testing['date'],model2_assessment['model2_predicted_revenue'],marker='o',label='model 2')

plt.legend()
plt.xticks(rotation=45)
plt.show()


**To:** H&S Management  
**From:** Analyst  
**Subject:** Revenue Forecasting Model Comparison

Two forecasting models were developed to predict H&S's revenue. Model 1 used production, a Summer dummy variable, and an interaction term between production and Summer. Model 2 used coolDD, a Spring dummy variable, and an interaction term between coolDD and Spring.

The models were evaluated using the testing data and compared using mean absolute percentage error. Model 1 produced a MAPE of 19.68%, while Model 2 produced a MAPE of 26.62%. Since a lower MAPE indicates better forecasting accuracy, Model 1 performed better.

Based on these results, H&S should use Model 1 to forecast revenue. Future analysis could test whether adding other explanatory variables further improves forecasting accuracy.
